In [37]:
import face_alignment
import cv2
import os
import pandas as pd
import numpy as np
import torch
import shutil # Thư viện để copy file
from tqdm import tqdm
from datetime import datetime

# --- CẤU HÌNH ---
INPUT_FOLDERS = [
    "128_crop_dataset_yunet",
    "128_crop_dataset_using_default_bbox"
]

OUTPUT_CSV = "landmark_dataset_clean_fan.csv"

# Cấu hình thư mục chứa ảnh lỗi
ERROR_DIR = "failed_landmark_images"
ERROR_LOG_FILE = "processing_errors.log"

# ĐỊNH NGHĨA CÁC ĐIỂM CẦN GIỮ LẠI (17-67)
SELECTED_INDICES = list(range(17, 68))

def log_error(filename, reason, source_path):
    """
    Hàm hỗ trợ: Ghi log và copy ảnh lỗi sang thư mục riêng
    """
    # 1. Tạo thư mục lỗi nếu chưa có
    if not os.path.exists(ERROR_DIR):
        os.makedirs(ERROR_DIR)

    # 2. Ghi vào file log text
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    log_path = os.path.join(ERROR_DIR, ERROR_LOG_FILE)
    
    with open(log_path, "a", encoding="utf-8") as f:
        f.write(f"[{timestamp}] {filename}: {reason}\n")

    # 3. Copy file ảnh gốc sang thư mục lỗi để kiểm tra thủ công
    try:
        # Giữ nguyên tên file, copy vào thư mục ERROR_DIR
        destination = os.path.join(ERROR_DIR, filename)
        
        # Nếu file đã tồn tại trong thư mục lỗi (do trùng tên từ thư mục khác), thêm tiền tố
        if os.path.exists(destination):
            name, ext = os.path.splitext(filename)
            destination = os.path.join(ERROR_DIR, f"{name}_copy{ext}")
            
        shutil.copy2(source_path, destination)
    except Exception as e:
        print(f" -> Không thể copy file lỗi {filename}: {e}")

def main():
    # 1. Khởi tạo Face Alignment
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"--- KHỞI TẠO ---")
    print(f"Thiết bị: {device.upper()}")
    print(f"Thư mục lỗi: {os.path.abspath(ERROR_DIR)}")
    
    try:
        fa = face_alignment.FaceAlignment(face_alignment.LandmarksType.TWO_D, 
                                          device=device, 
                                          face_detector='sfd')
    except Exception as e:
        print(f"Lỗi khởi tạo Face Alignment: {e}")
        return

    data_list = []
    
    # Tạo header CSV
    columns = ["image_id"]
    for i in range(len(SELECTED_INDICES)):
        columns.extend([f"x_{i}", f"y_{i}"])

    # 2. Duyệt ảnh
    total_processed = 0
    total_failed = 0
    
    # Xóa file log cũ nếu muốn bắt đầu mới (tùy chọn)
    if os.path.exists(os.path.join(ERROR_DIR, ERROR_LOG_FILE)):
        open(os.path.join(ERROR_DIR, ERROR_LOG_FILE), 'w').close()

    for folder in INPUT_FOLDERS:
        if not os.path.exists(folder):
            print(f"Bỏ qua thư mục không tồn tại: {folder}")
            continue
            
        files = [f for f in os.listdir(folder) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]
        print(f"Đang xử lý thư mục '{folder}' - {len(files)} ảnh...")

        for filename in tqdm(files, unit="img"):
            img_path = os.path.join(folder, filename)
            
            # --- KIỂM TRA 1: Đọc ảnh ---
            img = cv2.imread(img_path)
            if img is None:
                log_error(filename, "OpenCV không đọc được file (file hỏng hoặc sai định dạng)", img_path)
                total_failed += 1
                continue
            
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            
            try:
                # --- KIỂM TRA 2: Dự đoán Landmark ---
                preds = fa.get_landmarks(img_rgb)
                
                if preds is None or len(preds) == 0:
                    log_error(filename, "FAN không tìm thấy khuôn mặt (ảnh mờ, quá tối hoặc crop sai)", img_path)
                    total_failed += 1
                    continue
                
                # Lấy kết quả
                landmarks = preds[0] 

                # Trích xuất toạ độ
                row = [filename]
                for idx in SELECTED_INDICES:
                    px = landmarks[idx][0]
                    py = landmarks[idx][1]
                    row.extend([px, py])
                
                data_list.append(row)
                total_processed += 1
                
            except Exception as e:
                # --- KIỂM TRA 3: Lỗi Runtime ---
                log_error(filename, f"Lỗi ngoại lệ trong quá trình xử lý: {str(e)}", img_path)
                total_failed += 1
                continue

    # 3. Lưu kết quả
    print("\n--- KẾT THÚC ---")
    if len(data_list) > 0:
        df = pd.DataFrame(data_list, columns=columns)
        df = df.sort_values("image_id")
        df.to_csv(OUTPUT_CSV, index=False)
        print(f"Đã lưu dữ liệu landmark vào: {OUTPUT_CSV}")
    else:
        print("CẢNH BÁO: Không có dữ liệu nào được trích xuất thành công.")

    print(f"Tổng số ảnh thành công: {total_processed}")
    print(f"Tổng số ảnh thất bại: {total_failed}")
    if total_failed > 0:
        print(f"Vui lòng kiểm tra thư mục '{ERROR_DIR}' để xem các ảnh lỗi.")

if __name__ == "__main__":
    main()

--- KHỞI TẠO ---
Thiết bị: CUDA
Thư mục lỗi: e:\Sao lưu ổ C\File học\Năm 4\Dự án khoa học\failed_landmark_images
Đang xử lý thư mục '128_crop_dataset_yunet' - 202507 ảnh...


  1%|          | 1398/202507 [01:32<3:08:10, 17.81img/s]C:\Users\tavie\AppData\Roaming\Python\Python312\site-packages\face_alignment\api.py:147: UserWarning: No faces were detected.
  warnings.warn("No faces were detected.")
100%|██████████| 202507/202507 [3:08:10<00:00, 17.94img/s]  


Đang xử lý thư mục '128_crop_dataset_using_default_bbox' - 92 ảnh...


100%|██████████| 92/92 [00:03<00:00, 24.73img/s]



--- KẾT THÚC ---
Đã lưu dữ liệu landmark vào: landmark_dataset_clean_fan.csv
Tổng số ảnh thành công: 202528
Tổng số ảnh thất bại: 71
Vui lòng kiểm tra thư mục 'failed_landmark_images' để xem các ảnh lỗi.


In [1]:
import torch
import sys

print("Python version:", sys.version)
print("PyTorch version:", torch.__version__)
print("--------------------------------------------------")

# Kiểm tra CUDA
is_cuda = torch.cuda.is_available()
print(f"CUDA available: {is_cuda}")

if is_cuda:
    print(f"Số lượng GPU: {torch.cuda.device_count()}")
    print(f"Tên GPU: {torch.cuda.get_device_name(0)}")
    print(f"Phiên bản CUDA mà PyTorch đang dùng: {torch.version.cuda}")
    
    # Test thử một phép tính trên GPU
    try:
        x = torch.tensor([1.0, 2.0]).cuda()
        print("Test Tensor trên GPU thành công:", x)
    except Exception as e:
        print("Lỗi khi đẩy dữ liệu vào GPU:", e)
else:
    print("!!! CẢNH BÁO: PyTorch vẫn chưa nhìn thấy GPU !!!")
    print("Gợi ý: Kiểm tra lại xem bạn đã 'pip uninstall' bản cũ chưa, hoặc driver quá cũ.")

Python version: 3.12.3 (tags/v3.12.3:f6650f9, Apr  9 2024, 14:05:25) [MSC v.1938 64 bit (AMD64)]
PyTorch version: 2.5.1+cu121
--------------------------------------------------
CUDA available: True
Số lượng GPU: 1
Tên GPU: NVIDIA GeForce GTX 1650
Phiên bản CUDA mà PyTorch đang dùng: 12.1
Test Tensor trên GPU thành công: tensor([1., 2.], device='cuda:0')


In [48]:
import pandas as pd
import cv2
import os
import random
import numpy as np

# --- CẤU HÌNH ---
CSV_FILE = "landmark_dataset_clean_fan.csv"  # File kết quả của bạn
INPUT_FOLDERS = [
    "128_crop_dataset_using_default_bbox", "128_crop_dataset_yunet"
]
OUTPUT_DIR = "verification_results" # Thư mục chứa ảnh kiểm tra
NUM_SAMPLES = 20 # Số lượng ảnh muốn kiểm tra ngẫu nhiên

def draw_landmarks(img, row, num_points=51):
    """Hàm vẽ điểm lên ảnh"""
    img_copy = img.copy()
    h, w = img.shape[:2]
    
    for i in range(num_points):
        # Tên cột trong CSV là x_0, y_0, ..., x_50, y_50
        col_x = f"x_{i}"
        col_y = f"y_{i}"
        
        if col_x in row and col_y in row:
            x = float(row[col_x])
            y = float(row[col_y])
            
            # Vẽ chấm tròn màu xanh lá
            # Lưu ý: OpenCV dùng BGR -> (0, 255, 0) là xanh lá
            cv2.circle(img_copy, (int(x), int(y)), 1, (0, 255, 0), -1)
            
    return img_copy

def main():
    # 1. Đọc dữ liệu
    print(f"Đang đọc file {CSV_FILE}...")
    try:
        df = pd.read_csv(CSV_FILE)
    except FileNotFoundError:
        print("Lỗi: Không tìm thấy file CSV!")
        return

    print(f"Tổng số dữ liệu: {len(df)}")
    
    # 2. Tạo thư mục output
    if not os.path.exists(OUTPUT_DIR):
        os.makedirs(OUTPUT_DIR)

    # 3. Lấy mẫu ngẫu nhiên
    samples = df.sample(n=min(NUM_SAMPLES, len(df)))
    
    print("Đang vẽ landmark lên ảnh mẫu...")
    count = 0
    
    for idx, row in samples.iterrows():
        image_name = "002626.jpg"
        
        # Tìm đường dẫn ảnh gốc
        img_path = None
        for folder in INPUT_FOLDERS:
            temp_path = os.path.join(folder, image_name)
            if os.path.exists(temp_path):
                img_path = temp_path
                break
        
        if img_path:
            img = cv2.imread(img_path)
            if img is not None:
                # Vẽ landmark
                # Số lượng điểm dựa trên số cột: (tổng cột - 1 cột id) / 2
                num_points = (len(df.columns) - 1) // 2
                result_img = draw_landmarks(img, row, num_points)
                
                # Lưu kết quả
                save_path = os.path.join(OUTPUT_DIR, f"verify_{image_name}")
                cv2.imwrite(save_path, result_img)
                count += 1
            else:
                print(f"Không đọc được ảnh: {image_name}")
        else:
            print(f"Không tìm thấy ảnh gốc: {image_name}")

    print(f"\nĐã hoàn tất! Hãy mở thư mục '{OUTPUT_DIR}' để kiểm tra.")
    print("Tiêu chí kiểm tra:")
    print("1. Các chấm xanh có bám sát mắt, mũi, miệng không?")
    print("2. Có chấm nào bị bay ra khỏi khuôn mặt không?")

if __name__ == "__main__":
    main()

Đang đọc file landmark_dataset_clean_fan.csv...
Tổng số dữ liệu: 202528
Đang vẽ landmark lên ảnh mẫu...

Đã hoàn tất! Hãy mở thư mục 'verification_results' để kiểm tra.
Tiêu chí kiểm tra:
1. Các chấm xanh có bám sát mắt, mũi, miệng không?
2. Có chấm nào bị bay ra khỏi khuôn mặt không?


In [13]:
import os
from tqdm import tqdm

# --- CẤU HÌNH ---
FAILED_DIR = "failed_landmark_images"  # Thư mục chứa các ảnh lỗi bạn đã lọc ra
SOURCE_FOLDERS = [
    "128_crop_dataset_yunet",
    "128_crop_dataset_using_default_bbox"
]

# [QUAN TRỌNG] Đổi thành False để xóa thật. Để True để chạy thử (chỉ in ra màn hình).
DRY_RUN = False 

def main():
    # 1. Lấy danh sách tên các file bị lỗi
    if not os.path.exists(FAILED_DIR):
        print(f"Lỗi: Không tìm thấy thư mục '{FAILED_DIR}'")
        return

    # Chỉ lấy tên file (ví dụ: 000001.jpg), không cần đường dẫn đầy đủ
    failed_filenames = [f for f in os.listdir(FAILED_DIR) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]
    
    print(f"--- BẮT ĐẦU DỌN DẸP ---")
    print(f"Tìm thấy {len(failed_filenames)} ảnh trong thư mục lỗi '{FAILED_DIR}'.")
    if DRY_RUN:
        print("CHẾ ĐỘ: CHẠY THỬ (DRY RUN) - Chưa có file nào bị xóa.")
    else:
        print("CHẾ ĐỘ: XÓA THẬT (LIVE) - Các file trùng khớp sẽ bị xóa vĩnh viễn!")

    deleted_count = 0
    not_found_count = 0

    # 2. Duyệt qua từng file lỗi và tìm nó trong các thư mục nguồn để xóa
    for filename in tqdm(failed_filenames, unit="img"):
        file_deleted = False
        
        for source_folder in SOURCE_FOLDERS:
            # Tạo đường dẫn đầy đủ đến file trong thư mục nguồn
            target_path = os.path.join(source_folder, filename)
            
            # Kiểm tra xem file này có tồn tại trong thư mục nguồn không
            if os.path.exists(target_path):
                if not DRY_RUN:
                    try:
                        os.remove(target_path) # Lệnh xóa file
                        file_deleted = True
                    except Exception as e:
                        print(f"Không thể xóa {target_path}: {e}")
                else:
                    # Nếu là chạy thử, coi như đã xóa để đếm
                    file_deleted = True
        
        if file_deleted:
            deleted_count += 1
        else:
            not_found_count += 1

    # 3. Tổng kết
    print("\n--- KẾT QUẢ ---")
    if DRY_RUN:
        print(f"Dự kiến sẽ xóa: {deleted_count} ảnh khỏi các thư mục nguồn.")
        print("Hãy đổi DRY_RUN = False trong code và chạy lại để xóa thật.")
    else:
        print(f"Đã xóa thành công: {deleted_count} ảnh.")
        print("Thư mục nguồn của bạn hiện đã sạch sẽ!")

if __name__ == "__main__":
    main()

--- BẮT ĐẦU DỌN DẸP ---
Tìm thấy 71 ảnh trong thư mục lỗi 'failed_landmark_images'.
CHẾ ĐỘ: XÓA THẬT (LIVE) - Các file trùng khớp sẽ bị xóa vĩnh viễn!


100%|██████████| 71/71 [00:00<00:00, 1991.73img/s]


--- KẾT QUẢ ---
Đã xóa thành công: 71 ảnh.
Thư mục nguồn của bạn hiện đã sạch sẽ!


In [42]:
import pandas as pd
import cv2
import os
from tqdm import tqdm

# --- CẤU HÌNH ---
# File CSV chuẩn nhất của bạn (đã clip giá trị âm)
INPUT_CSV = "landmark_dataset_clean_fan.csv"

# Thư mục chứa ảnh sạch (đã gom ở bước trước)
IMAGE_FOLDER = "128_crop_dataset_yunet"

# Thư mục sẽ lưu ảnh đã vẽ landmark (để bạn xem)
OUTPUT_DIR = "visualized_dataset"

# [QUAN TRỌNG] Số lượng ảnh muốn vẽ. 
# Đặt = None nếu muốn vẽ TOÀN BỘ (hơn 200k ảnh - Cẩn thận đầy ổ cứng!)
# Đặt = 200 để kiểm tra nhanh.
LIMIT_NUM = 2000 

def main():
    # 1. Đọc dữ liệu
    if not os.path.exists(INPUT_CSV):
        print(f"Lỗi: Không tìm thấy file {INPUT_CSV}")
        return
    
    print(f"Đang đọc dữ liệu từ {INPUT_CSV}...")
    df = pd.read_csv(INPUT_CSV)
    total_images = len(df)
    
    # 2. Xử lý giới hạn số lượng
    if LIMIT_NUM is not None and LIMIT_NUM < total_images:
        print(f"Chế độ: Lấy mẫu ngẫu nhiên {LIMIT_NUM} ảnh để kiểm tra.")
        # Lấy mẫu ngẫu nhiên (random state để tái lập kết quả)
        process_df = df.sample(n=LIMIT_NUM, random_state=556)
    else:
        print(f"Chế độ: Xử lý TOÀN BỘ {total_images} ảnh.")
        process_df = df

    # 3. Tạo thư mục output
    if not os.path.exists(OUTPUT_DIR):
        os.makedirs(OUTPUT_DIR)
        print(f"Đã tạo thư mục kết quả: {OUTPUT_DIR}")

    # 4. Vẽ landmark
    print("Đang tiến hành vẽ landmark...")
    
    # Xác định số lượng điểm dựa trên số cột
    # (Tổng cột - 1 cột tên ảnh) chia 2
    num_landmarks = (len(df.columns) - 1) // 2
    
    success_count = 0
    
    for idx, row in tqdm(process_df.iterrows(), total=len(process_df), unit="img"):
        filename = row['image_id']
        img_path = os.path.join(IMAGE_FOLDER, filename)
        
        if not os.path.exists(img_path):
            continue
            
        img = cv2.imread(img_path)
        if img is None:
            continue
            
        # Vẽ từng điểm
        for i in range(num_landmarks):
            # Lấy toạ độ từ cột x_0, y_0, ...
            # Lưu ý: Ép kiểu về int để vẽ
            try:
                x = int(float(row[f'x_{i}']))
                y = int(float(row[f'y_{i}']))
                
                # Vẽ chấm tròn đặc (thickness = -1), bán kính 1, màu Xanh Lá (0, 255, 0)
                cv2.circle(img, (x, y), 1, (0, 255, 0), -1)
            except ValueError:
                pass
        
        # Lưu ảnh sang thư mục mới
        save_path = os.path.join(OUTPUT_DIR, filename)
        cv2.imwrite(save_path, img)
        success_count += 1

    print(f"\n--- HOÀN TẤT ---")
    print(f"Đã lưu {success_count} ảnh vào thư mục '{OUTPUT_DIR}'.")
    print("Hãy vào thư mục đó và lướt nhanh để kiểm tra chất lượng.")

if __name__ == "__main__":
    main()

Đang đọc dữ liệu từ landmark_dataset_clean_fan.csv...
Chế độ: Lấy mẫu ngẫu nhiên 2000 ảnh để kiểm tra.
Đang tiến hành vẽ landmark...


100%|██████████| 2000/2000 [03:13<00:00, 10.32img/s]


--- HOÀN TẤT ---
Đã lưu 2000 ảnh vào thư mục 'visualized_dataset'.
Hãy vào thư mục đó và lướt nhanh để kiểm tra chất lượng.


In [51]:
import pandas as pd
import os
import shutil
from tqdm import tqdm

# --- CẤU HÌNH ---
# Hãy chắc chắn bạn dùng file gôc (CHƯA qua bước kẹp/clipping về 0)
INPUT_CSV = "landmark_dataset_clean_fan.csv" 
IMAGE_FOLDER = "128_crop_dataset_using_default_bbox" # Thư mục chứa ảnh gốc

# File CSV chứa kết quả lọc
OUTPUT_CSV = "negative_landmarks.csv"
# Thư mục để copy ảnh vào xem cho tiện
DEBUG_FOLDER = "debug_negative_images"

def main():
    # 1. Đọc dữ liệu
    if not os.path.exists(INPUT_CSV):
        print(f"Lỗi: Không tìm thấy file {INPUT_CSV}")
        return

    print(f"Đang đọc file {INPUT_CSV}...")
    df = pd.read_csv(INPUT_CSV)
    
    # Lấy danh sách các cột toạ độ (x_... và y_...)
    coord_cols = [c for c in df.columns if c.startswith('x_') or c.startswith('y_')]
    
    # 2. Lọc các hàng có giá trị âm
    # Logic: Kiểm tra từng dòng, nếu có BẤT KỲ (any) cột nào < 0 thì giữ lại
    print("Đang quét tìm giá trị âm...")
    condition = (df[coord_cols] < 0).any(axis=1)
    negative_df = df[condition]
    
    # Thống kê
    total_negative = len(negative_df)
    print(f"-> Tìm thấy {total_negative} ảnh chứa landmark âm (trên tổng số {len(df)} ảnh).")

    if total_negative == 0:
        print("Tuyệt vời! Không có landmark nào bị âm. Bạn không cần làm gì thêm.")
        return

    # 3. Lưu ra file CSV riêng
    negative_df.to_csv(OUTPUT_CSV, index=False)
    print(f"-> Đã lưu danh sách vào file: {OUTPUT_CSV}")

    # 4. Copy ảnh ra thư mục riêng để kiểm tra mắt thường (Visual Inspection)
    print(f"Đang copy ảnh sang thư mục '{DEBUG_FOLDER}' để bạn kiểm tra...")
    
    if not os.path.exists(DEBUG_FOLDER):
        os.makedirs(DEBUG_FOLDER)
        
    # Giới hạn copy khoảng 50 ảnh để xem mẫu thôi (tránh copy quá nhiều nếu số lượng lớn)
    # Nếu muốn copy hết thì bỏ dòng [:50] đi
    sample_to_copy = negative_df.head(50) 
    
    count = 0
    for filename in tqdm(sample_to_copy['image_id'], unit="img"):
        src_path = os.path.join(IMAGE_FOLDER, filename)
        dst_path = os.path.join(DEBUG_FOLDER, filename)
        
        if os.path.exists(src_path):
            shutil.copy2(src_path, dst_path)
            count += 1
            
    print(f"\n--- HOÀN TẤT ---")
    print(f"Đã copy {count} ảnh mẫu vào thư mục '{DEBUG_FOLDER}'.")
    print("Hãy mở thư mục đó ra xem. Bạn sẽ thấy hầu hết là các ảnh chụp góc nghiêng (profile) hoặc ảnh crop sát mặt.")

if __name__ == "__main__":
    main()

Đang đọc file landmark_dataset_clean_fan.csv...
Đang quét tìm giá trị âm...
-> Tìm thấy 118 ảnh chứa landmark âm (trên tổng số 202528 ảnh).
-> Đã lưu danh sách vào file: negative_landmarks.csv
Đang copy ảnh sang thư mục 'debug_negative_images' để bạn kiểm tra...


100%|██████████| 50/50 [00:00<00:00, 24989.90img/s]


--- HOÀN TẤT ---
Đã copy 1 ảnh mẫu vào thư mục 'debug_negative_images'.
Hãy mở thư mục đó ra xem. Bạn sẽ thấy hầu hết là các ảnh chụp góc nghiêng (profile) hoặc ảnh crop sát mặt.


In [52]:
import pandas as pd
import numpy as np

# --- CẤU HÌNH ---
INPUT_CSV = "landmark_dataset_clean_fan.csv"
OUTPUT_CSV = "landmark_dataset_final_clipped.csv"
IMAGE_SIZE = 128

def main():
    print(f"Đang đọc file {INPUT_CSV}...")
    df = pd.read_csv(INPUT_CSV)
    
    # Lấy danh sách các cột toạ độ (bắt đầu bằng x_ hoặc y_)
    coord_cols = [c for c in df.columns if c.startswith('x_') or c.startswith('y_')]
    
    print("Thống kê trước khi sửa:")
    print(f"Min value: {df[coord_cols].min().min()}")
    print(f"Max value: {df[coord_cols].max().max()}")
    
    # Thực hiện CLIPPING (Kẹp giá trị trong khoảng [0, IMAGE_SIZE])
    # clip(lower, upper) sẽ thay thế các giá trị < 0 bằng 0 và > 128 bằng 128
    df[coord_cols] = df[coord_cols].clip(0, IMAGE_SIZE)
    
    print("-" * 30)
    print("Thống kê sau khi sửa:")
    print(f"Min value: {df[coord_cols].min().min()}")
    print(f"Max value: {df[coord_cols].max().max()}")
    
    # Lưu file mới
    df.to_csv(OUTPUT_CSV, index=False)
    print(f"\nĐã lưu file sạch tại: {OUTPUT_CSV}")
    print("Hãy dùng file này cho bước huấn luyện L-Gen và Inverse Mapping!")

if __name__ == "__main__":
    main()

Đang đọc file landmark_dataset_clean_fan.csv...
Thống kê trước khi sửa:
Min value: -54.0
Max value: 155.0
------------------------------
Thống kê sau khi sửa:
Min value: 0.0
Max value: 128.0

Đã lưu file sạch tại: landmark_dataset_final_clipped.csv
Hãy dùng file này cho bước huấn luyện L-Gen và Inverse Mapping!


In [57]:
import pandas as pd
import cv2
import os
import numpy as np
from tqdm import tqdm

# --- CẤU HÌNH ---
# 1. File CSV gốc (chứa số âm) - Output của bước face-alignment
ORIGINAL_CSV = "landmark_dataset_clean_fan.csv"

# 2. File CSV đã sửa (đã clipping) - Output của bước sửa lỗi
CLIPPED_CSV = "landmark_dataset_final_clipped.csv"

# 3. Thư mục ảnh
IMAGE_FOLDER = "128_crop_dataset_yunet"

# 4. Thư mục xuất ảnh so sánh
OUTPUT_DIR = "comparison_results"

# Kích thước lề đệm thêm để nhìn thấy điểm âm (pixel)
PADDING = 50 

def get_landmarks(row):
    """Trích xuất danh sách các điểm (x, y) từ một hàng CSV"""
    points = []
    # Xác định số lượng điểm dựa trên số cột
    # (Tổng cột - 1 cột tên ảnh) / 2
    num_points = (len(row) - 1) // 2
    
    for i in range(num_points):
        try:
            x = float(row[f'x_{i}'])
            y = float(row[f'y_{i}'])
            points.append((x, y))
        except KeyError:
            pass
    return points

def main():
    # 1. Kiểm tra file
    if not os.path.exists(ORIGINAL_CSV) or not os.path.exists(CLIPPED_CSV):
        print("Lỗi: Thiếu file CSV đầu vào.")
        return

    print("Đang đọc dữ liệu...")
    df_orig = pd.read_csv(ORIGINAL_CSV).set_index("image_id")
    df_clip = pd.read_csv(CLIPPED_CSV).set_index("image_id")

    # 2. Tìm các ảnh có toạ độ âm trong file gốc để đối chiếu
    # Lấy các cột toạ độ
    coord_cols = [c for c in df_orig.columns if c.startswith('x_') or c.startswith('y_')]
    
    # Lọc ra danh sách các ảnh có ít nhất 1 giá trị < 0
    negative_rows = df_orig[(df_orig[coord_cols] < 0).any(axis=1)]
    target_filenames = negative_rows.index.tolist()

    print(f"Tìm thấy {len(target_filenames)} ảnh có landmark âm để đối chiếu.")
    
    # 3. Tạo thư mục output
    if not os.path.exists(OUTPUT_DIR):
        os.makedirs(OUTPUT_DIR)

    # 4. Vẽ đối chiếu
    print(f"Đang vẽ ảnh so sánh vào thư mục '{OUTPUT_DIR}'...")
    
    # Giới hạn vẽ 50 ảnh mẫu để kiểm tra (bỏ [:50] nếu muốn vẽ hết)
    for filename in tqdm(target_filenames[:50], unit="img"):
        img_path = os.path.join(IMAGE_FOLDER, filename)
        
        if not os.path.exists(img_path):
            continue
            
        img = cv2.imread(img_path)
        if img is None:
            continue

        # --- TẠO CANVAS LỚN HƠN ---
        # Thêm viền đen xung quanh ảnh để nhìn thấy điểm âm
        # copyMakeBorder(src, top, bottom, left, right, borderType, value)
        canvas = cv2.copyMakeBorder(img, PADDING, PADDING, PADDING, PADDING, 
                                    cv2.BORDER_CONSTANT, value=[50, 50, 50]) # Màu xám đậm

        # Lấy toạ độ từ 2 file
        try:
            pts_orig = get_landmarks(df_orig.loc[filename])
            pts_clip = get_landmarks(df_clip.loc[filename])
        except KeyError:
            continue

        # Vẽ điểm
        for i in range(len(pts_orig)):
            # Toạ độ gốc (có thể âm)
            ox, oy = pts_orig[i]
            # Toạ độ clip (đã kẹp về 0-128)
            cx, cy = pts_clip[i]

            # Dịch chuyển toạ độ theo PADDING để vẽ lên canvas mới
            # Ví dụ: x cũ là -10 -> x vẽ là -10 + 50 = 40 (nằm trong vùng đệm)
            draw_ox, draw_oy = int(ox + PADDING), int(oy + PADDING)
            draw_cx, draw_cy = int(cx + PADDING), int(cy + PADDING)

            # Chỉ vẽ sự khác biệt (những điểm bị thay đổi)
            # Dùng khoảng cách nhỏ để so sánh float
            if abs(ox - cx) > 0.01 or abs(oy - cy) > 0.01:
                
                # 1. Vẽ đường nối (Màu Vàng)
                cv2.line(canvas, (draw_ox, draw_oy), (draw_cx, draw_cy), (0, 255, 255), 1)

                # 2. Vẽ điểm Gốc (Màu Đỏ - Red) -> Điểm bị lỗi/âm
                cv2.circle(canvas, (draw_ox, draw_oy), 2, (0, 0, 255), -1)
                
                # 3. Vẽ điểm Clip (Màu Xanh Lá - Green) -> Điểm đã sửa
                cv2.circle(canvas, (draw_cx, draw_cy), 2, (0, 255, 0), -1)

            else:
                # Những điểm không thay đổi thì vẽ màu xanh dương nhạt cho đỡ rối
                cv2.circle(canvas, (draw_cx, draw_cy), 1, (255, 0, 0), -1)

        # Vẽ khung hình chữ nhật biểu thị kích thước ảnh gốc 128x128
        cv2.rectangle(canvas, (PADDING, PADDING), 
                      (PADDING + 128, PADDING + 128), (255, 255, 255), 1)

        # Lưu ảnh
        cv2.imwrite(os.path.join(OUTPUT_DIR, f"compare_{filename}"), canvas)

    print("\n--- HOÀN TẤT ---")
    print("Giải thích màu sắc trong ảnh:")
    print("- Khung trắng: Phạm vi ảnh gốc (128x128)")
    print("- Chấm ĐỎ: Toạ độ gốc (bị âm hoặc nằm ngoài ảnh)")
    print("- Chấm XANH LÁ: Toạ độ mới (đã kẹp vào mép ảnh)")
    print("- Đường VÀNG: Sự dịch chuyển do thuật toán clipping")

if __name__ == "__main__":
    main()

Đang đọc dữ liệu...
Tìm thấy 118 ảnh có landmark âm để đối chiếu.
Đang vẽ ảnh so sánh vào thư mục 'comparison_results'...


100%|██████████| 50/50 [00:00<00:00, 89.46img/s]


--- HOÀN TẤT ---
Giải thích màu sắc trong ảnh:
- Khung trắng: Phạm vi ảnh gốc (128x128)
- Chấm ĐỎ: Toạ độ gốc (bị âm hoặc nằm ngoài ảnh)
- Chấm XANH LÁ: Toạ độ mới (đã kẹp vào mép ảnh)
- Đường VÀNG: Sự dịch chuyển do thuật toán clipping


In [59]:
import pandas as pd
import os
import datetime
from tqdm import tqdm

# --- CẤU HÌNH ---
# 1. Các thư mục và file đầu vào
CSV_FILE = "landmark_dataset_clean_fan.csv"
SOURCE_FOLDERS = [
    "128_crop_dataset_using_default_bbox",
    "128_crop_dataset_yunet"
]
FAILED_DIR = "failed_landmark_images"

# 2. File đầu ra
NEW_CSV_FILE = "landmark_dataset_strictly_positive.csv" # File CSV mới sạch 100%
LOG_FILE = "deletion_log.txt"

# [QUAN TRỌNG] Đổi thành False để XÓA THẬT. Để True để chạy thử (chỉ in log).
DRY_RUN = False

def log_deletion(f_handle, filename, reason, folder_found):
    """Hàm hỗ trợ ghi log"""
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    msg = f"[{timestamp}] DELETE: '{filename}' | FOUND_IN: '{folder_found}' | REASON: {reason}\n"
    f_handle.write(msg)
    # print(msg.strip()) # Bỏ comment nếu muốn in ra màn hình

def delete_file_from_sources(filename, source_folders, dry_run=True):
    """Tìm và xóa file trong danh sách thư mục nguồn"""
    deleted_paths = []
    
    for folder in source_folders:
        file_path = os.path.join(folder, filename)
        if os.path.exists(file_path):
            if not dry_run:
                try:
                    os.remove(file_path)
                    deleted_paths.append(folder)
                except Exception as e:
                    print(f"Lỗi không xoá được {file_path}: {e}")
            else:
                deleted_paths.append(folder) # Giả vờ đã xoá
                
    return deleted_paths

def main():
    if not os.path.exists(CSV_FILE):
        print(f"Lỗi: Không tìm thấy file CSV '{CSV_FILE}'")
        return

    # Mở file log
    log_mode = 'a' if os.path.exists(LOG_FILE) else 'w'
    with open(LOG_FILE, log_mode, encoding='utf-8') as log_f:
        log_f.write(f"\n--- SESSION STARTED: {datetime.datetime.now()} ---\n")
        log_f.write(f"MODE: {'DRY RUN (Chạy thử)' if DRY_RUN else 'LIVE (Xóa thật)'}\n")
        
        print(f"Đang đọc dữ liệu từ {CSV_FILE}...")
        df = pd.read_csv(CSV_FILE)
        initial_count = len(df)
        
        # --- BƯỚC 1: XÁC ĐỊNH ẢNH CẦN XÓA TỪ CSV (Landmark âm) ---
        coord_cols = [c for c in df.columns if c.startswith('x_') or c.startswith('y_')]
        
        # Tìm các dòng có ít nhất 1 giá trị < 0
        negative_mask = (df[coord_cols] < 0).any(axis=1)
        negative_df = df[negative_mask]
        negative_filenames = set(negative_df['image_id'].tolist())
        
        print(f"-> Tìm thấy {len(negative_filenames)} ảnh có landmark âm trong CSV.")

        # --- BƯỚC 2: XÁC ĐỊNH ẢNH CẦN XÓA TỪ THƯ MỤC FAILED ---
        failed_filenames = set()
        if os.path.exists(FAILED_DIR):
            failed_filenames = set([f for f in os.listdir(FAILED_DIR) if f.lower().endswith(('.jpg', '.png', '.jpeg'))])
        
        print(f"-> Tìm thấy {len(failed_filenames)} ảnh trong thư mục Failed.")
        
        # Gộp chung danh sách cần xóa (dùng set để loại bỏ trùng lặp nếu có)
        all_files_to_delete = negative_filenames.union(failed_filenames)
        print(f"-> TỔNG CỘNG: {len(all_files_to_delete)} file cần xử lý.")
        
        # --- BƯỚC 3: THỰC HIỆN XÓA TRONG THƯ MỤC NGUỒN ---
        print("Đang tiến hành quét và xóa ảnh trong thư mục nguồn...")
        deleted_count = 0
        
        for filename in tqdm(all_files_to_delete, unit="img"):
            # Xác định nguyên nhân
            reasons = []
            if filename in negative_filenames:
                reasons.append("Landmark âm")
            if filename in failed_filenames:
                reasons.append("Nằm trong thư mục Failed")
            reason_str = " & ".join(reasons)
            
            # Gọi hàm xóa
            folders_found = delete_file_from_sources(filename, SOURCE_FOLDERS, dry_run=DRY_RUN)
            
            # Ghi log nếu tìm thấy và xóa được
            if folders_found:
                deleted_count += 1
                for folder in folders_found:
                    log_deletion(log_f, filename, reason_str, folder)

        # --- BƯỚC 4: TẠO FILE CSV MỚI (LỌC BỎ DÒNG ÂM) ---
        if not DRY_RUN:
            # Chỉ giữ lại các dòng KHÔNG nằm trong negative_mask
            # Lưu ý: Ta không cần lọc failed_filenames khỏi CSV vì failed_filenames 
            # vốn dĩ không có trong CSV này (do CSV được tạo từ những lần chạy thành công).
            # Tuy nhiên, để chắc chắn, ta lọc những dòng có ID nằm trong negative_filenames.
            
            clean_df = df[~df['image_id'].isin(negative_filenames)]
            clean_df.to_csv(NEW_CSV_FILE, index=False)
            print(f"-> Đã lưu file CSV sạch mới: {NEW_CSV_FILE}")
            print(f"   Số dòng ban đầu: {initial_count}")
            print(f"   Số dòng còn lại: {len(clean_df)}")
            print(f"   Số dòng bị loại bỏ: {initial_count - len(clean_df)}")

    # --- TỔNG KẾT ---
    print("\n--- HOÀN TẤT ---")
    if DRY_RUN:
        print(f"DRY RUN: Sẽ có {deleted_count} file bị xóa khỏi thư mục nguồn.")
        print("Hãy đổi biến DRY_RUN = False trong code để thực hiện xóa thật.")
    else:
        print(f"Đã xóa thành công {deleted_count} file ảnh.")
        print(f"Chi tiết xem tại file log: {LOG_FILE}")
        print(f"Hãy sử dụng file CSV mới: '{NEW_CSV_FILE}' cho quá trình training.")

if __name__ == "__main__":
    main()

Đang đọc dữ liệu từ landmark_dataset_clean_fan.csv...
-> Tìm thấy 118 ảnh có landmark âm trong CSV.
-> Tìm thấy 71 ảnh trong thư mục Failed.
-> TỔNG CỘNG: 189 file cần xử lý.
Đang tiến hành quét và xóa ảnh trong thư mục nguồn...


100%|██████████| 189/189 [00:00<00:00, 712.64img/s]


-> Đã lưu file CSV sạch mới: landmark_dataset_strictly_positive.csv
   Số dòng ban đầu: 202528
   Số dòng còn lại: 202410
   Số dòng bị loại bỏ: 118

--- HOÀN TẤT ---
Đã xóa thành công 189 file ảnh.
Chi tiết xem tại file log: deletion_log.txt
Hãy sử dụng file CSV mới: 'landmark_dataset_strictly_positive.csv' cho quá trình training.
